In [ ]:
import os
import stim
import tqecd
import itertools

from tqec import NoiseModel

from utils.error_rate_analyser import analyse_error_rates

In [ ]:
touched = { 'N' : (0.0, -1.0), 'E' : (-1.0, 0.0), 'S' : (0.0, +1.0), 'W' : (+1.0, 0.0) }

def visit(schedule):
    if len(schedule) != 4 and set(schedule) != set(touched.keys()):
        raise ValueError("Schedule must be a permutation of { N,E,W,S }.")

    return list(map(lambda d : touched[d], schedule))

def usc_filename(distance, rounds, schedule):
    return f"../assets/surface-code/unrotated-surface-code-d{distance}-r{rounds}-s{schedule}.stim"

def unrotated_surface_code(distance = 3, rounds=2, schedule = "NEWS", savefile = False):
    side = 2*distance - 1
    dq_at_locations = dict()
    mq_at_locations = dict()
    xs = set()
    zs = set()

    usc = stim.Circuit()

    qubit = 0
    for location in itertools.product(range(side), range(side)):
        x, y = location

        usc.append(f"QUBIT_COORDS", qubit, location)
        if x % 2 == y % 2:
            dq_at_locations[location] = qubit
        else:
            mq_at_locations[location] = qubit
            if y % 2 == 0:
                xs.add(qubit)
            else:
                zs.add(qubit)
        qubit += 1

    usc.append("TICK")

    usc.append("RZ", dq_at_locations.values())

    usc.append("TICK")

    for round in range(rounds):
        usc.append("RX", mq_at_locations.values())
        usc.append("TICK")

        for moment in range(4):
            dx, dy = visit(schedule)[moment]
            cxs = []
            czs = []
            for (mx, my), mq in mq_at_locations.items():
                if 0 <= mx + dx < side and 0 <= my + dy < side:
                    if mq in xs:
                        cxs.extend( [ mq, dq_at_locations[(mx + dx, my + dy)] ])
                    else:
                        czs.extend( [ mq, dq_at_locations[(mx + dx, my + dy)] ])
            usc.append("CX", cxs)
            usc.append("CZ", czs)
        usc.append("TICK")

        usc.append("MX", mq_at_locations.values())
        usc.append("TICK")

    usc.append("MZ", dq_at_locations.values())

    if savefile:
        filename = usc_filename(distance, rounds, schedule)
        usc.to_file(filename)

        # Insert all the polygons into the Stim file for readability.
        with open(filename, "r", encoding="utf-8") as file:
            circuit_lines = file.readlines()
            insertion = 0

            # Find end of QUBIT_COORDS declarations
            while circuit_lines[insertion].startswith("QUBIT_COORDS"):
                insertion += 1

            for location, mq in mq_at_locations.items():
                x, y = location
                polygon = []
                for dx, dy in visit("NESW"):
                    vx, vy = x + dx, y + dy
                    if 0 <= vx < side and 0 <= vy < side:
                        polygon.append( str(dq_at_locations[vx, vy]) )
                circuit_lines.insert(insertion, f"#!pragma POLYGON({int(mq in xs)},0,{int(mq in zs)},0.5) {" ".join(polygon)}\n")
                insertion += 1

        with open(filename, "w", encoding="utf-8") as file:
            file.writelines(circuit_lines)

    return usc

In [ ]:
distance = 3
rounds = 2
schedule = "NEWS"

circuit = unrotated_surface_code(distance, rounds, schedule, savefile=True)
filename = filename = usc_filename(distance, rounds, schedule)
basename, _ = os.path.splitext(filename)
circuit = stim.Circuit().from_file(filename)
annotated = basename + ".annotated.stim"
if not os.path.exists(annotated):
    tqecd.annotate_detectors_automatically(circuit).to_file(annotated)

In [ ]:
circuit = stim.Circuit().from_file(annotated)

noisy = NoiseModel.uniform_depolarizing(0.01).noisy_circuit(circuit)
errors = noisy.shortest_graphlike_error(canonicalize_circuit_errors=True)

print(f"Length of shortest graph-like error : {len(errors)}")
for error in errors:
    print(error)

noisy.diagram("matchgraph-3d")

In [ ]:
analyse_error_rates(circuit, name=basename)

In [ ]:
distance = 3
rounds = 2
schedule = "NSEW"

circuit = unrotated_surface_code(distance, rounds, schedule, savefile=True)
filename = filename = usc_filename(distance, rounds, schedule)
basename, _ = os.path.splitext(filename)
circuit = stim.Circuit().from_file(filename)
annotated = basename + ".annotated.stim"
if not os.path.exists(annotated):
    tqecd.annotate_detectors_automatically(circuit).to_file(annotated)

In [ ]:
circuit = stim.Circuit().from_file(annotated)

noisy = NoiseModel.uniform_depolarizing(0.01).noisy_circuit(circuit)
errors = noisy.shortest_graphlike_error(canonicalize_circuit_errors=True)

print(f"Length of shortest graph-like error : {len(errors)}")
for error in errors:
    print(error)

print(f"Missing detectors : {len(noisy.missing_detectors())}")

noisy.diagram("matchgraph-3d")

In [ ]:
analyse_error_rates(circuit, name=basename)